In [49]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re 


In [50]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [51]:

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 KDD 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        doi_link = doi_tag["href"] if doi_tag else None

        # PDF URL 생성
        if doi_link:
            pdf_link = doi_link.replace("doi.org", "dl.acm.org/doi/pdf")
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            "code_url" :None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [52]:
url = 'https://dblp.org/db/conf/mm/mm2024.html'
DB_PATH = "con_db/MM_conference_2024.db"
conference_name = 'MM 2024'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [53]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2024_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [54]:
df_papers = get_www_papers('html/MM_2024_accepted_papers.html', conference_name)

In [55]:
df_papers.head(15)

,title,authors,pdf_link,code_url,conference_name
0,From Assistants to Agents in the LLM Era.,Pascale Fung,https://dl.acm.org/doi/pdf/10.1145/3664647.367...,None,MM 2024
1,Revolutionizing Lung Cancer Diagnostics with e...,Benoit Huet,https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
2,Empowering People to Harness and Control their...,Judy Kay,https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
3,Large Multimodal Models as Social Multimedia A...,Jiebo Luo,https://dl.acm.org/doi/pdf/10.1145/3664647.367...,None,MM 2024
4,"When, Where, and What? A Benchmark for Acciden...","Haicheng Liao, Yongkang Li, Chengyue Wang, Yan...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
5,A Unified Understanding of Adversarial Vulnera...,"Haonan Zheng, Xinyang Deng, Wen Jiang, Wenrui Li",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
6,Not All Inputs Are Valid: Towards Open-Set Vid...,"Xiang Fang, Wanlong Fang, Daizong Liu, Xiaoye ...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
7,Towards Flexible Evaluation for Generative Vis...,"Huishan Ji, Qingyi Si, Zheng Lin, Weiping Wang",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
8,Do LLMs Understand Visual Anomalies? Uncoverin...,"Jiaqi Zhu, Shaofeng Cai, Fang Deng, Beng Chin ...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
9,FLIP-80M: 80 Million Visual-Linguistic Pairs f...,"Yudong Li, Xianxu Hou, Dezhi Zheng, Linlin She...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024


In [56]:
df_papers = df_papers.drop(index=[0,1,2,3,4])

In [57]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
5,A Unified Understanding of Adversarial Vulnera...,"Haonan Zheng, Xinyang Deng, Wen Jiang, Wenrui Li",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
6,Not All Inputs Are Valid: Towards Open-Set Vid...,"Xiang Fang, Wanlong Fang, Daizong Liu, Xiaoye ...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
7,Towards Flexible Evaluation for Generative Vis...,"Huishan Ji, Qingyi Si, Zheng Lin, Weiping Wang",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
8,Do LLMs Understand Visual Anomalies? Uncoverin...,"Jiaqi Zhu, Shaofeng Cai, Fang Deng, Beng Chin ...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
9,FLIP-80M: 80 Million Visual-Linguistic Pairs f...,"Yudong Li, Xianxu Hou, Dezhi Zheng, Linlin She...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024


In [58]:
df_papers.tail()

,title,authors,pdf_link,code_url,conference_name
1232,MEGC2024: ACM Multimedia 2024 Facial Micro-Exp...,"John See, Jingting Li, Adrian K. Davison, Gen-...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
1233,Temporal-Informative Adapters in VideoMAE V2 a...,"Jun Yu, Gongpeng Zhao, Yaohui Zhang, Peng He, ...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
1234,Micro-Expression Spotting Based on Optical Flo...,"Jun Yu, Yaohui Zhang, Gongpeng Zhao, Peng He, ...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
1235,A Multi-scale Feature Learning Network with Op...,"Zhengye Zhang, Sirui Zhao, Xinglong Mao, Shife...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024
1236,Enhancing Micro-Expression Analysis Performanc...,"Yuhong He, Wenchao Liu, Guangyu Wang, Lin Ma, ...",https://dl.acm.org/doi/pdf/10.1145/3664647.368...,None,MM 2024


In [59]:
save_to_database(df_papers, conference_name, DB_PATH)

1232개의 논문이 MM 2024에 저장되었습니다.


# ACM MM 2023

In [60]:
url = 'https://dblp.org/db/conf/mm/mm2023.html'
DB_PATH = "con_db/MM_conference_2023.db"
conference_name = 'MM 2023'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [61]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2023_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [62]:
df_papers = get_www_papers('html/MM_2023_accepted_papers.html', conference_name)

In [63]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Internet of Video Things: Technical Challenges...,Chang Wen Chen,https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
1,Multimodal AI & LLMs for Peacekeeping and Emer...,Alejandro Jaimes,https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
2,Transition and Adaptability: The Cornerstone o...,Ralf Steinmetz,https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
3,Mutual Information-driven Triple Interaction N...,"Hao Shen, Zhong-Qiu Zhao, Yulun Zhang, Zhao Zhang",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
4,Suspected Objects Matter: Rethinking Model's P...,"Yang Jiao, Zequn Jie, Jingjing Chen, Lin Ma, Y...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023


In [64]:
df_papers = df_papers.drop(index=[0,1,2,3])

In [65]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
4,Suspected Objects Matter: Rethinking Model's P...,"Yang Jiao, Zequn Jie, Jingjing Chen, Lin Ma, Y...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
5,Self-Relational Graph Convolution Network for ...,"Sophyani Banaamwini Yussif, Ning Xie, Yang Yan...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
6,Exploring Correlations in Degraded Spatial Ide...,"Qian Ning, Fangfang Wu, Weisheng Dong, Xin Li,...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
7,Video-based Visible-Infrared Person Re-Identif...,"Chuhao Zhou, Jinxing Li, Huafeng Li, Guangming...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
8,PetalView: Fine-grained Location and Orientati...,"Wenmiao Hu, Yichen Zhang, Yuxuan Liang, Xianji...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023


In [66]:
df_papers.tail()

,title,authors,pdf_link,code_url,conference_name
1009,HCMA '23: 4th International Workshop on Human-...,"Jingkuan Song, Wu Liu, Xinchen Liu, Dingwen Zh...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
1010,FME '23: 3rd Facial Micro-Expression Workshop.,"Adrian K. Davison, Jingting Li, Moi Hoon Yap, ...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
1011,Deep Multimodal Learning for Information Retri...,"Wei Ji, Yinwei Wei, Zhedong Zheng, Hao Fei, Ta...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
1012,AMC-SME '23: 2023 Workshop on Advanced Multime...,"Junxin Chen, Wei Wang, Gwanggil Jeon",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023
1013,LGM3A '23: 1st Workshop on Large Generative Mo...,"Zheng Wang, Cheng Long, Shihao Xu, Bingzheng G...",https://dl.acm.org/doi/pdf/10.1145/3581783.361...,None,MM 2023


In [67]:
save_to_database(df_papers, conference_name, DB_PATH)

1010개의 논문이 MM 2023에 저장되었습니다.


# ACM 2022

In [72]:
url = 'https://dblp.org/db/conf/mm/mm2022.html'
DB_PATH = "con_db/MM_conference_2022.db"
conference_name = 'MM 2022'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [73]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2022_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [74]:
df_papers = get_www_papers('html/MM_2022_accepted_papers.html', conference_name)

In [75]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,"Alexa, let's work together! How Alexa Helps Cu...",Yoelle Maarek,https://dl.acm.org/doi/pdf/10.1145/3503161.354...,None,MM 2022
1,Data Science against COVID-19: The Valencian E...,Nuria Oliver,https://dl.acm.org/doi/pdf/10.1145/3503161.354...,None,MM 2022
2,"Grounding, Meaning and Foundation Models: Adve...",Douwe Kiela,https://dl.acm.org/doi/pdf/10.1145/3503161.354...,None,MM 2022
3,A Multi-view Spectral-Spatial-Temporal Masked ...,"Rui Li, Yiting Wang, Wei-Long Zheng, Bao-Liang Lu",https://dl.acm.org/doi/pdf/10.1145/3503161.354...,None,MM 2022
4,Counterfactual Reasoning for Out-of-distributi...,"Teng Sun, Wenjie Wang, Liqiang Jing, Yiran Cui...",https://dl.acm.org/doi/pdf/10.1145/3503161.354...,None,MM 2022


In [76]:
df_papers = df_papers.drop(index=[0,1,2])

In [78]:
df_papers.tail(20)

,title,authors,pdf_link,code_url,conference_name
809,Memory Networks.,"Federico Becattini, Tiberio Uricchio",https://dl.acm.org/doi/pdf/10.1145/3503161.354...,None,MM 2022
810,Open Challenges of Interactive Video Search an...,"Jakub Lokoc, Klaus Schoeffmann, Werner Bailer,...",https://dl.acm.org/doi/pdf/10.1145/3503161.354...,None,MM 2022
811,MMSports'22: 5th International ACM Workshop on...,"Hideo Saito, Thomas B. Moeslund, Rainer Lienhart",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022
812,"MuSe 2022 Challenge: Multimodal Humour, Emotio...","Shahin Amiriparian, Lukas Christ, Andreas Köni...",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022
813,APCCPA '22: 1st International Workshop on Adva...,"Wei Gao, Ge Li, Hui Yuan, Raouf Hamzaoui, Zhu ...",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022
814,M4MM '22: 1st International Workshop on Method...,"Xavier Alameda-Pineda, Qin Jin, Vincent Oria, ...",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022
815,FME '22: 2nd Workshop on Facial Micro-Expressi...,"Jingting Li, Moi Hoon Yap, Wen-Huang Cheng, Jo...",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022
816,NarSUM '22: 1st Workshop on User-centric Narra...,"Mohan S. Kankanhalli, Jianquan Liu, Yongkang W...",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022
817,CEA++'22: 1st International Workshop on Multim...,"Yoko Yamakata, Atsushi Hashimoto, Jingjing Chen",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022
818,DDAM '22: 1st International Workshop on Deepfa...,"Jianhua Tao, Jiangyan Yi, Cunhang Fan, Ruibo F...",https://dl.acm.org/doi/pdf/10.1145/3503161.355...,None,MM 2022


In [80]:
save_to_database(df_papers, conference_name, DB_PATH)

826개의 논문이 MM 2022에 저장되었습니다.


# ACM 2021

In [82]:
url = 'https://dblp.org/db/conf/mm/mm2021.html'
DB_PATH = "con_db/MM_conference_2021.db"
conference_name = 'MM 2021'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [83]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2021_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [84]:
df_papers = get_www_papers('html/MM_2021_accepted_papers.html', conference_name)

In [85]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Video Coding for Machine.,Wen Gao,https://dl.acm.org/doi/pdf/10.1145/3474085.348...,None,MM 2021
1,Semantic Media Conversion: Possibilities and L...,H. V. Jagadish,https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021
2,Image Re-composition via Regional Content-Styl...,"Rong Zhang, Wei Li, Yiqun Zhang, Hong Zhang, J...",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021
3,Deep Clustering based on Bi-Space Association ...,"Hao Huang, Shinjae Yoo, Chenxiao Xu",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021
4,Feature Stylization and Domain-aware Contrasti...,"Seogkyu Jeon, Kibeom Hong, Pilhyeon Lee, Jewoo...",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021


In [86]:
df_papers = df_papers.drop(index=[0,1])

In [87]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
2,Image Re-composition via Regional Content-Styl...,"Rong Zhang, Wei Li, Yiqun Zhang, Hong Zhang, J...",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021
3,Deep Clustering based on Bi-Space Association ...,"Hao Huang, Shinjae Yoo, Chenxiao Xu",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021
4,Feature Stylization and Domain-aware Contrasti...,"Seogkyu Jeon, Kibeom Hong, Pilhyeon Lee, Jewoo...",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021
5,HDA-Net: Horizontal Deformable Attention Netwo...,"Qi Zhang, Xuesong Zhang, Baoping Li, Yuzhong C...",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021
6,MBRS: Enhancing Robustness of DNN-based Waterm...,"Zhaoyang Jia, Han Fang, Weiming Zhang",https://dl.acm.org/doi/pdf/10.1145/3474085.347...,None,MM 2021


In [88]:
save_to_database(df_papers, conference_name, DB_PATH)

682개의 논문이 MM 2021에 저장되었습니다.


# ACM 2020

In [89]:
url = 'https://dblp.org/db/conf/mm/mm2020.html'
DB_PATH = "con_db/MM_conference_2020.db"
conference_name = 'MM 2020'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [90]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2020_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [91]:
df_papers = get_www_papers('html/MM_2020_accepted_papers.html', conference_name)

In [92]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Image Inpainting Based on Multi-frequency Prob...,"Jin Wang, Chen Wang, Qingming Huang, Yunhui Sh...",https://dl.acm.org/doi/pdf/10.1145/3394171.341...,None,MM 2020
1,Dual Adversarial Network for Unsupervised Grou...,"Jianzhe Lin, Lichao Mou, Tianze Yu, Xiaoxiang ...",https://dl.acm.org/doi/pdf/10.1145/3394171.341...,None,MM 2020
2,Adversarial Bipartite Graph Learning for Video...,"Yadan Luo, Zi Huang, Zijian Wang, Zheng Zhang,...",https://dl.acm.org/doi/pdf/10.1145/3394171.341...,None,MM 2020
3,Give Me Something to Eat: Referring Expression...,"Peng Wang, Dongyang Liu, Hui Li, Qi Wu",https://dl.acm.org/doi/pdf/10.1145/3394171.341...,None,MM 2020
4,Single Image De-noising via Staged Memory Netw...,"Weijiang Yu, Jian Liang, Lu Li, Nong Xiao",https://dl.acm.org/doi/pdf/10.1145/3394171.341...,None,MM 2020


In [93]:
save_to_database(df_papers, conference_name, DB_PATH)

584개의 논문이 MM 2020에 저장되었습니다.


# ACM 2019

In [96]:
url = 'https://dblp.org/db/conf/mm/mm2019.html'
DB_PATH = "con_db/MM_conference_2019.db"
conference_name = 'MM 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [97]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [98]:
df_papers = get_www_papers('html/MM_2019_accepted_papers.html', conference_name)

In [99]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Using Artificial Intelligence to Preserve Audi...,Jean Carrive,https://dl.acm.org/doi/pdf/10.1145/3343031.334...,None,MM 2019
1,Focus Your Attention: A Bidirectional Focal At...,"Chunxiao Liu, Zhendong Mao, An-An Liu, Tianzhu...",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019
2,Matching Images and Text with Multi-modal Tens...,"Tan Wang, Xing Xu, Yang Yang, Alan Hanjalic, H...",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019
3,Structured Stochastic Recurrent Network for Li...,"Shijie Yang, Liang Li, Shuhui Wang, Dechao Men...",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019
4,Visual Relationship Detection with Relative Lo...,"Hao Zhou, Chongyang Zhang, Chuanping Hu",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019


In [100]:
df_papers = df_papers.drop(index=[0])

In [101]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Focus Your Attention: A Bidirectional Focal At...,"Chunxiao Liu, Zhendong Mao, An-An Liu, Tianzhu...",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019
2,Matching Images and Text with Multi-modal Tens...,"Tan Wang, Xing Xu, Yang Yang, Alan Hanjalic, H...",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019
3,Structured Stochastic Recurrent Network for Li...,"Shijie Yang, Liang Li, Shuhui Wang, Dechao Men...",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019
4,Visual Relationship Detection with Relative Lo...,"Hao Zhou, Chongyang Zhang, Chuanping Hu",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019
5,Vision-Language Recommendation via Attribute A...,"Tong Yu, Yilin Shen, Ruiyi Zhang, Xiangyu Zeng...",https://dl.acm.org/doi/pdf/10.1145/3343031.335...,None,MM 2019


In [102]:
save_to_database(df_papers, conference_name, DB_PATH)

370개의 논문이 MM 2019에 저장되었습니다.


# ACM MM 2018

In [103]:
url = 'https://dblp.org/db/conf/mm/mm2018.html'
DB_PATH = "con_db/MM_conference_2018.db"
conference_name = 'MM 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [104]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [105]:
df_papers = get_www_papers('html/MM_2018_accepted_papers.html', conference_name)

In [106]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Session details: FF-1.,Max Mühlhäuser,None,None,MM 2018
1,SCRATCH: A Scalable Discrete Matrix Factorizat...,"Chuan-Xiang Li, Zhen-Duo Chen, Peng-Fei Zhang,...",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018
2,Predicting Visual Context for Unsupervised Eve...,"Ana Garcia del Molino, Joo-Hwee Lim, Ah-Hwee Tan",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018
3,Video-to-Video Translation with Global Tempora...,"Xingxing Wei, Jun Zhu, Sitong Feng, Hang Su",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018
4,Shared Linear Encoder-based Gaussian Process L...,"Jinxing Li, Bob Zhang, Guangming Lu, David Zhang",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018


In [107]:
df_papers = df_papers.drop(index=[0])

In [108]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,SCRATCH: A Scalable Discrete Matrix Factorizat...,"Chuan-Xiang Li, Zhen-Duo Chen, Peng-Fei Zhang,...",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018
2,Predicting Visual Context for Unsupervised Eve...,"Ana Garcia del Molino, Joo-Hwee Lim, Ah-Hwee Tan",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018
3,Video-to-Video Translation with Global Tempora...,"Xingxing Wei, Jun Zhu, Sitong Feng, Hang Su",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018
4,Shared Linear Encoder-based Gaussian Process L...,"Jinxing Li, Bob Zhang, Guangming Lu, David Zhang",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018
5,"Step-by-step Erasion, One-by-one Collection: A...","Jia-Xing Zhong, Nannan Li, Weijie Kong, Tao Zh...",https://dl.acm.org/doi/pdf/10.1145/3240508.324...,None,MM 2018


In [109]:
save_to_database(df_papers, conference_name, DB_PATH)

312개의 논문이 MM 2018에 저장되었습니다.


# ACM MM 2017

In [110]:
url = 'https://dblp.org/db/conf/mm/mm2017.html'
DB_PATH = "con_db/MM_conference_2017.db"
conference_name = 'MM 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [111]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [113]:
df_papers = get_www_papers('html/MM_2017_accepted_papers.html', conference_name)

In [114]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Attention Transfer from Web Images for Video R...,"Junnan Li, Yongkang Wong, Qi Zhao, Mohan S. Ka...",https://dl.acm.org/doi/pdf/10.1145/3123266.312...,None,MM 2017
1,SketchParse: Towards Rich Descriptions for Poo...,"Ravi Kiran Sarvadevabhatla, Isht Dwivedi, Abhi...",https://dl.acm.org/doi/pdf/10.1145/3123266.312...,None,MM 2017
2,Place-centric Visual Urban Perception with Dee...,"Xiaobai Liu, Qi Chen, Lei Zhu, Yuanlu Xu, Lian...",https://dl.acm.org/doi/pdf/10.1145/3123266.312...,None,MM 2017
3,Future-Supervised Retrieval of Unseen Queries ...,"Spencer Cappallo, Cees G. M. Snoek",https://dl.acm.org/doi/pdf/10.1145/3123266.312...,None,MM 2017
4,Learning to Compose with Professional Photogra...,"Yi-Ling Chen, Jan Klopp, Min Sun, Shao-Yi Chie...",https://dl.acm.org/doi/pdf/10.1145/3123266.312...,None,MM 2017


In [115]:
save_to_database(df_papers, conference_name, DB_PATH)

273개의 논문이 MM 2017에 저장되었습니다.


# ACM MM 2016

In [116]:
url = 'https://dblp.org/db/conf/mm/mm2016.html'
DB_PATH = "con_db/MM_conference_2016.db"
conference_name = 'MM 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [117]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [118]:
df_papers = get_www_papers('html/MM_2016_accepted_papers.html', conference_name)

In [119]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,A Digital World to Thrive In: How the Internet...,Dirk Helbing,https://dl.acm.org/doi/pdf/10.1145/2964284.298...,None,MM 2016
1,Multi-modal Multi-view Topic-opinion Mining fo...,"Shengsheng Qian, Tianzhu Zhang, Changsheng Xu",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016
2,Patterns of Free-form Curation: Visual Thinkin...,"Nic Lupfer, Andruid Kerne, Andrew M. Webb, Rhe...",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016
3,DASH2M: Exploring HTTP/2 for Internet Streamin...,"Mengbai Xiao, Viswanathan Swaminathan, Sheng W...",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016
4,Deep-based Ingredient Recognition for Cooking ...,"Jingjing Chen, Chong-Wah Ngo",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016


In [120]:
df_papers = df_papers.drop(index=[0])

In [121]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Multi-modal Multi-view Topic-opinion Mining fo...,"Shengsheng Qian, Tianzhu Zhang, Changsheng Xu",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016
2,Patterns of Free-form Curation: Visual Thinkin...,"Nic Lupfer, Andruid Kerne, Andrew M. Webb, Rhe...",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016
3,DASH2M: Exploring HTTP/2 for Internet Streamin...,"Mengbai Xiao, Viswanathan Swaminathan, Sheng W...",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016
4,Deep-based Ingredient Recognition for Cooking ...,"Jingjing Chen, Chong-Wah Ngo",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016
5,GeoTracks: Adaptive Music for Everyday Journeys.,"Chris Greenhalgh, Adrian Hazzard, Sean McGrath...",https://dl.acm.org/doi/pdf/10.1145/2964284.296...,None,MM 2016


In [122]:
save_to_database(df_papers, conference_name, DB_PATH)

270개의 논문이 MM 2016에 저장되었습니다.


# ACM MM 2015

In [123]:
url = 'https://dblp.org/db/conf/mm/mm2015.html'
DB_PATH = "con_db/MM_conference_2015.db"
conference_name = 'MM 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [124]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [125]:
df_papers = get_www_papers('html/MM_2015_accepted_papers.html', conference_name)

In [126]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,"Harnessing Big Personal Data, with Scrutable U...",Judy Kay,https://dl.acm.org/doi/pdf/10.1145/2733373.280...,None,MM 2015
1,Vision-enhanced Immersive Interaction and Remo...,Zhengyou Zhang,https://dl.acm.org/doi/pdf/10.1145/2733373.281...,None,MM 2015
2,Analyzing Free-standing Conversational Groups:...,"Xavier Alameda-Pineda, Yan Yan, Elisa Ricci, O...",https://dl.acm.org/doi/pdf/10.1145/2733373.280...,None,MM 2015
3,An Affordable Solution for Binocular Eye Track...,"Michael Stengel, Steve Grogorick, Martin Eisem...",https://dl.acm.org/doi/pdf/10.1145/2733373.280...,None,MM 2015
4,SINGA: Putting Deep Learning in the Hands of M...,"Wei Wang, Gang Chen, Tien Tuan Anh Dinh, Jinya...",https://dl.acm.org/doi/pdf/10.1145/2733373.280...,None,MM 2015


In [127]:
df_papers =  df_papers.drop(index=[0,1])

In [128]:
save_to_database(df_papers, conference_name, DB_PATH)

270개의 논문이 MM 2015에 저장되었습니다.


# ACM MM 2014

In [129]:
url = 'https://dblp.org/db/conf/mm/mm2014.html'
DB_PATH = "con_db/MM_conference_2014.db"
conference_name = 'MM 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [130]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/MM_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [131]:
df_papers = get_www_papers('html/MM_2014_accepted_papers.html', conference_name)

In [132]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,"Bing, the fastest growing image search engine.",Harry Shum,https://dl.acm.org/doi/pdf/10.1145/2647868.264...,None,MM 2014
1,Affective media and wearables: surprising find...,Rosalind W. Picard,https://dl.acm.org/doi/pdf/10.1145/2647868.264...,None,MM 2014
2,Back and to the future: quality provisioning f...,Klara Nahrstedt,https://dl.acm.org/doi/pdf/10.1145/2647868.266...,None,MM 2014
3,Cross-modal Retrieval with Correspondence Auto...,"Fangxiang Feng, Xiaojie Wang, Ruifan Li",https://dl.acm.org/doi/pdf/10.1145/2647868.265...,None,MM 2014
4,VideoStory: A New Multimedia Embedding for Few...,"AmirHossein Habibian, Thomas Mensink, Cees G. ...",https://dl.acm.org/doi/pdf/10.1145/2647868.265...,None,MM 2014


In [133]:
df_papers = df_papers.drop(index=[0,1,2])

In [134]:
save_to_database(df_papers, conference_name, DB_PATH)

248개의 논문이 MM 2014에 저장되었습니다.


In [135]:
df_papers['pdf_link'].head()

3    https://dl.acm.org/doi/pdf/10.1145/2647868.265...
4    https://dl.acm.org/doi/pdf/10.1145/2647868.265...
5    https://dl.acm.org/doi/pdf/10.1145/2647868.265...
6    https://dl.acm.org/doi/pdf/10.1145/2647868.265...
7    https://dl.acm.org/doi/pdf/10.1145/2647868.265...
Name: pdf_link, dtype: object